# Handle Data Storage

### Create a list of all Zettel

In [1]:
import os
import regex as re
import pandas as pd

from zettelsortierung.Datatypes import *
from zettelsortierung.Transformation import *
from zettelsortierung.RegionDetection import OpenCVRegionDetector
from zettelsortierung.OCR import OCR, OCRpreprocessing, OCRpostprocessing
from zettelsortierung.Visualisation import vis_boxes_labels

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
root = '../data/processed/Scans.parquet'
sammlung = Zettelsammlung.from_parquet(root, k=10)

detector = OpenCVRegionDetector()

box_probe = Probe()
text_probe = Probe()

pipeline = \
Composition(
    Batch(1000),
    SequentialApp(
        ParallelApp(
            PatchDetect(detector)
        ),
        Flatten(),
        ParallelApp(
            BoundingBoxPad(10)
        ),
        Store(box_probe),
        ParallelApp(
            CutOutPatch(),
            ResizePatch(),
            BGR2RGB()
        ),
        Sort(key=lambda item: item.feature.shape[1]),
        Batch(16),
        ParallelApp(OCRpreprocessing()),
        SequentialApp(OCR()),
        ParallelApp(OCRpostprocessing()),
        Flatten(),
        Store(text_probe)
    ),
    Flatten()
)

In [3]:
res = pipeline(sammlung)

In [4]:
print(box_probe)
print(len(box_probe))
print(text_probe)
print(len(text_probe))

Probe([DataPoint(zettel=Zettel(A01-00000043), feature_id=3, feature=BoundingBox(x=np.int32(906), y=np.int32(818), w=np.int32(130), h=np.int32(104))),
DataPoint(zettel=Zettel(A01-00000065), feature_id=0, feature=BoundingBox(x=np.int32(1689), y=np.int32(591), w=np.int32(183), h=np.int32(91))),
DataPoint(zettel=Zettel(A01-00000065), feature_id=1, feature=BoundingBox(x=np.int32(1201), y=np.int32(1124), w=np.int32(508), h=np.int32(97))),
DataPoint(zettel=Zettel(A01-00000039), feature_id=0, feature=BoundingBox(x=np.int32(1487), y=np.int32(1088), w=np.int32(379), h=np.int32(88))),
DataPoint(zettel=Zettel(A01-00000001), feature_id=0, feature=BoundingBox(x=np.int32(1551), y=np.int32(1028), w=np.int32(245), h=np.int32(90))),
DataPoint(zettel=Zettel(A01-00000059), feature_id=0, feature=BoundingBox(x=np.int32(1312), y=np.int32(991), w=np.int32(406), h=np.int32(175))),
DataPoint(zettel=Zettel(A01-00000085), feature_id=1, feature=BoundingBox(x=np.int32(971), y=np.int32(497), w=np.int32(887), h=np.in

In [5]:
# Build lookup dictionary from second list
text_lookup = {(zettel, feat_id): text for zettel, feat_id, text in text_probe}

# Join
pairs = [
    (zettel, feat_id, box, text_lookup[(zettel, feat_id)])
    for zettel, feat_id, box in box_probe
    if (zettel, feat_id) in text_lookup
]

In [ ]:
counter = 0
number = 1
zettel_set = set([dp.zettel for dp in box_probe])
for zettel in zettel_set:
    if counter >= number:
        break
    boxes = []
    labels = []
    for pair in pairs:
        if pair[0] == zettel:
            boxes.append(pair[2])
            labels.append(pair[3])
    vis_boxes_labels(zettel, boxes, labels)
    counter += 1


In [10]:
import sqlalchemy
from sqlalchemy import Column, Integer, String, ForeignKey, Table
from sqlalchemy.orm import relationship, backref
from sqlalchemy.ext.declarative import declarative_base

db_engine = sqlalchemy.create_engine('sqlite:///../data/processed/zettel.db')
connection = db_engine.connect()
metadata = sqlalchemy.MetaData()

In [11]:
print(metadata)

MetaData()
